# 🏆 FEATURE IMPORTANCE (ÖZELLİK ÖNEMİ)
> **🎯 AMAÇ:** Hangi feature'ların modeli en çok etkilediğini analiz etmek  
> **📥 GİRDİ:** Eğitilmiş model, X_train, X_test  
> **📤 ÇIKTI:** MDI + Permutation Importance grafikleri  
### 🔧 YÖNTEMLER
- **Tree-based MDI:** `model.feature_importances_`
- **Permutation:** Model agnostik, daha güvenilir
- **SHAP:** Yorumlanabilirlik standardı (opsiyonel)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
print('=' * 50 + '\n🏆 FEATURE IMPORTANCE BAŞLATILIYOR\n' + '=' * 50)

In [ ]:
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train.values.ravel())
feature_names = list(X_train.columns) if hasattr(X_train,'columns') else [f'f{i}' for i in range(X_train.shape[1])]

# MDI Importance
importances = model.feature_importances_
std = np.std([t.feature_importances_ for t in model.estimators_], axis=0)
df_imp = pd.DataFrame({'feature': feature_names, 'importance': importances, 'std': std})
df_imp = df_imp.sort_values('importance', ascending=False)
print('\n📊 MDI TOP-10:')
print(df_imp.head(10).to_string(index=False))

In [ ]:
# Permutation Importance
perm = permutation_importance(model, X_test, y_test, n_repeats=10, random_state=42, n_jobs=-1)
df_perm = pd.DataFrame({'feature': feature_names, 'perm_imp': perm.importances_mean, 'std': perm.importances_std})
df_perm = df_perm.sort_values('perm_imp', ascending=False)
print('\n📊 PERMUTATION TOP-10:')
print(df_perm.head(10).to_string(index=False))

In [ ]:
top_n = 15
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
top_mdi = df_imp.head(top_n)
colors = plt.cm.RdYlGn(np.linspace(0.3, 0.9, top_n))[::-1]
axes[0].barh(top_mdi['feature'][::-1], top_mdi['importance'][::-1], xerr=top_mdi['std'][::-1], color=colors, capsize=3)
axes[0].set_title('MDI Importance (Tree-based)', fontweight='bold')
top_perm = df_perm.head(top_n)
axes[1].barh(top_perm['feature'][::-1], top_perm['perm_imp'][::-1], xerr=top_perm['std'][::-1], color='#2196F3', capsize=3)
axes[1].set_title('Permutation Importance (Test set)', fontweight='bold')
plt.tight_layout(); plt.show()
# SHAP (pip install shap):
# import shap; explainer = shap.TreeExplainer(model)
# shap.summary_plot(explainer.shap_values(X_test)[1], X_test)
print('\n✅ İşlem Tamamlandı.')